In [ ]:
# Оценка поиска - после того как мы загрузили список вопросов, мы можем
# оценить, насколько хорошо наш поиск находит правильные документы. 

# Для каждого вопроса в нашем истинном наборе данных (ИНД) мы выполняем 
# поиск. Затем мы проверим, содержат ли результаты правильный документ.

In [23]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

# Фильтруем материалы только по llm-zoomcamp
for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

documents = documents_llm     # 153

# Создаём индекс для дальнейшего поиска по ним.
index = build_index(documents)   # <minsearch.minsearch.Index at 0x1722834f4d0>

In [24]:
def text_search(query):
  boost_dict = {"question": 3.0, "section": 0.5}

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict
  )

In [ ]:
import pandas as pd

# Читаем CSV-Файл и преобразуем следующей же строкой в словарь
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")


"\n{\n  'question': 'I found this course late — can I still enroll and follow along?',\n  'document': '74eb249bbf'\n}\n"

In [34]:
df_ground_truth.head()

,question,document
0,I found this course late — can I still enroll ...,74eb249bbf
1,"Is it too late to join the course now, or can ...",74eb249bbf
2,Can I still participate in the course even if ...,74eb249bbf
3,"If I join late, do I still have a chance to ge...",74eb249bbf
4,What’s the deadline if I want a certificate af...,74eb249bbf


In [ ]:
q = ground_truth[0]
q

'''
{
  'question': 'I found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'
}
'''

In [ ]:
# Обращаемся к id - документа для первого вопроса и выводим его => ground_truth[0]['document'] тоже
# самое что и вызвали раньше

doc_id = q["document"] 
# 74eb249bbf
doc_query = q["question"] 
# I found this course late — can I still enroll and follow along?

results = text_search(query = doc_query)

'''
[
  {
    'id': '04919992b3',
    'course': 'llm-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'How should I start the course and follow the weekly workflow?',
    'answer': 'Start with the LLM Zoomcamp docs the general Zoomcamp logistics docs
  },
  {
    'id': '74eb249bbf',
    'course': 'llm-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'I just discovered the course. Can I still join?',
    'answer': 
      'Yes, but if you want to receive a certificate, you need 
      to submit your project while we’re still accepting submissions.'
  }
  * 3
]
'''

"\n[\n  {\n    'id': '04919992b3',\n    'course': 'llm-zoomcamp',\n    'section': 'General Course-Related Questions',\n    'question': 'How should I start the course and follow the weekly workflow?',\n    'answer': 'Start with the LLM Zoomcamp docs the general Zoomcamp logistics docs\n  },\n  {\n    'id': '74eb249bbf',\n    'course': 'llm-zoomcamp',\n    'section': 'General Course-Related Questions',\n    'question': 'I just discovered the course. Can I still join?',\n    'answer': \n      'Yes, but if you want to receive a certificate, you need \n      to submit your project while we’re still accepting submissions.'\n  }\n  * 3\n]\n"

In [ ]:
# В начале сравниваем полученные id документов с настоящим id документов

for d in results:
  print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

'''
  04919992b3 == 74eb249bbf: False
  74eb249bbf == 74eb249bbf: True
  69d122f12e == 74eb249bbf: False
  a9353fadfe == 74eb249bbf: False
  85384a18e5 == 74eb249bbf: False
'''

04919992b3 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
69d122f12e == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False


In [ ]:
# Затем преобразуйте его в список релевантности (является ли полученный 
# документ правильным документом для данного вопроса.) 

relevance = []

for d in results:
  relevance.append(int(d["id"] == doc_id))

relevance

# [0, 1, 0, 0, 0]
# Дает список 0 значений 1, где 1 - означает, что полученный документ имеет 
# тот же id, что и истинный документ.

[0, 1, 0, 0, 0]

In [ ]:
# Все что сделали ранее добавим в одну общую функцию 
def compute_relevance_text(q):
  doc_id = q["document"]
  # doc_query = ground_truth[0]['question']
  results = text_search(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance

q = ground_truth[0]
print(doc_query)
compute_relevance_text(q)


# I found this course late — can I still enroll and follow along?
# [0, 1, 0, 0, 0]

I found this course late — can I still enroll and follow along?


[0, 1, 0, 0, 0]

In [ ]:
# Теперь проделаем то же самое со всеми вопросами
from tqdm.auto import tqdm

# функция считает метрику релевантности поиска для каждого вопроса из 
# истинного набора и собирает результаты в один список.
def compute_relevance_total_text(ground_truth):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance_text(q)
    relevance_total.append(relevance)

  return relevance_total

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

'''
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
  x*10
'''

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [33]:
# Каждая запись relevance_total_text представляет собой список релевантности. 
# Этого достаточно, чтобы проверить работоспособность функции, прежде чем 
# запускать её для всего набора данных. Далее, сделаем функции релевантности 
# универсальными. Логика релевантности остается той же. Меняется только 
# функция поиска.

def compute_relevance(q, search_function):
  doc_id = q["document"]
  # results = text_search(query=q["question"])
  results = search_function(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance


In [ ]:
def compute_relevance_total(ground_truth, search_function):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance(q, search_function)
    relevance_total.append(relevance)

  return relevance_total

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

  0%|          | 0/765 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,